# tok-adapt: multilingual end-to-end pipeline on an A100

Runs the full pipeline — **dedup → vocabulary expansion → CPT → SFT → DPO → evaluation → GGUF/ONNX export** —
against `Qwen/Qwen2.5-1.5B` and real Wikipedia text across a representative set of the world's
most-spoken languages.

**Honest framing before you run this:** this is still a *wiring proof at meaningfully larger scale* than the
3060 smoke test in the repo's README (real 1.5B model, real multilingual corpus, a full CPT epoch instead of
3 steps) — it is not a from-scratch training run to convergence. Expect a coherent, functioning pipeline and
believable-looking intermediate metrics; do not expect production-quality generation out the other end from
one epoch over tens of MB of text. Scale `NUM_ARTICLES_PER_LANG`, `ADD_VOCAB_SIZE`, and the `cpt`/`sft`/`dpo`
`num_train_epochs` up if you want to push further — the A100's ~40GB has a lot of headroom left after this.

**Runtime:** Runtime → Change runtime type → **A100 GPU**, then Runtime → Run all.

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Clone tok-adapt and install the full pipeline extras

If any imports fail right after this cell (rare, but Colab's preinstalled `numpy`/`transformers` can occasionally
clash with pinned versions), use **Runtime → Restart session** and re-run from here — don't re-run the clone.

In [ ]:
!git clone https://github.com/vishnup22/tok-adapt.git
%cd tok-adapt
!pip install -q -e ".[pipeline]"
!pip install -q datasets

## 3. (Optional) Confirm the test suite passes in this environment

Same 36 tests as the repo's own CI — a quick sanity check that nothing about Colab's specific package versions
broke anything before committing GPU time to the full run.

In [ ]:
!python -m pytest tests/ -q

## 4. Pull real text for the world's most-spoken languages

Streams a small number of articles per language from Wikipedia (`wikimedia/wikipedia` on the Hugging Face Hub)
— no download of the full dumps. `LANGUAGES` below is a representative top-12 by total speakers spanning
Latin, Han, Devanagari, Arabic, Bengali, Cyrillic, and Japanese scripts; extend the dict to add more.

In [ ]:
import random
from pathlib import Path
from datasets import load_dataset

LANGUAGES = {
    'en': 'English', 'zh': 'Mandarin Chinese', 'hi': 'Hindi', 'es': 'Spanish',
    'fr': 'French', 'ar': 'Arabic', 'bn': 'Bengali', 'ru': 'Russian',
    'pt': 'Portuguese', 'ur': 'Urdu', 'id': 'Indonesian', 'ja': 'Japanese',
}
WIKI_DUMP_DATE = '20231101'
NUM_ARTICLES_PER_LANG = 200   # raise for a larger corpus; A100 can handle it
MIN_PARAGRAPH_CHARS = 60

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
random.seed(0)

lang_paragraphs = {}
for code_, name in LANGUAGES.items():
    print(f'Fetching {name} ({code_})...')
    ds = load_dataset('wikimedia/wikipedia', f'{WIKI_DUMP_DATE}.{code_}', split='train', streaming=True)
    paras = []
    for i, article in enumerate(ds):
        if i >= NUM_ARTICLES_PER_LANG:
            break
        for para in article['text'].split('\n'):
            para = para.strip()
            if len(para) >= MIN_PARAGRAPH_CHARS:
                paras.append(para)
    lang_paragraphs[code_] = paras
    out_path = raw_dir / f'{code_}.txt'
    out_path.write_text('\n'.join(paras), encoding='utf-8')
    print(f'  -> {len(paras)} paragraphs, {out_path.stat().st_size / 1e6:.2f} MB')

## 5. Build SFT + DPO data

- **SFT**: plain-text continuation examples sampled directly from the fetched paragraphs (schema:
  `{"text": ...}` — see `tok_adapt.training.sft.SupervisedFineTuner._load_examples`).
- **DPO**: a genuine, non-fabricated preference signal for *this project's* exact purpose — for a leading
  sentence in language X, the **chosen** response is a real same-language continuation and the **rejected**
  response is a paragraph from a *different* language spliced in. This rewards staying in the prompt's
  language, which is directly relevant to cross-lingual adaptation quality.

In [ ]:
import json

data_dir = Path('data')
all_codes = list(lang_paragraphs.keys())

sft_records = []
for code_, paras in lang_paragraphs.items():
    sample = random.sample(paras, min(40, len(paras)))
    sft_records.extend({'text': p} for p in sample)
random.shuffle(sft_records)
(data_dir / 'sft_data.jsonl').write_text(
    '\n'.join(json.dumps(r, ensure_ascii=False) for r in sft_records), encoding='utf-8'
)
print(f'SFT: {len(sft_records)} examples -> data/sft_data.jsonl')

dpo_records = []
for code_, paras in lang_paragraphs.items():
    other_codes = [c for c in all_codes if c != code_]
    sample = random.sample(paras, min(30, len(paras)))
    for p in sample:
        words = p.split(' ')
        if len(words) < 8:
            continue
        split_at = len(words) // 3
        prompt = ' '.join(words[:split_at])
        chosen = ' '.join(words[split_at:])
        rejected_lang = random.choice(other_codes)
        rejected = random.choice(lang_paragraphs[rejected_lang])
        dpo_records.append({'prompt': prompt, 'chosen': chosen, 'rejected': rejected})
random.shuffle(dpo_records)
(data_dir / 'dpo_data.jsonl').write_text(
    '\n'.join(json.dumps(r, ensure_ascii=False) for r in dpo_records), encoding='utf-8'
)
print(f'DPO: {len(dpo_records)} preference triples -> data/dpo_data.jsonl')

eval_paras = []
for paras in lang_paragraphs.values():
    eval_paras.extend(random.sample(paras, min(15, len(paras))))
(data_dir / 'eval.txt').write_text('\n'.join(eval_paras), encoding='utf-8')
print(f'Eval: {len(eval_paras)} held-out paragraphs -> data/eval.txt')

## 6. Write the pipeline config

Base model is `Qwen/Qwen2.5-1.5B` (not the -Instruct variant — this pipeline performs its own CPT/SFT/DPO
on top of a base checkpoint). `gguf_pre_tokenizer_hint: qwen2` is required for GGUF export once the tokenizer
has been expanded — see the README's *Exporting expanded checkpoints to GGUF* section for why.

In [ ]:
import yaml

config = {
    'model': 'Qwen/Qwen2.5-1.5B',
    'output_root': './pipeline_out',
    'dedup': {
        'enabled': True,
        'input_paths': [str(p) for p in sorted(raw_dir.glob('*.txt'))],
        'languages': None,  # already known-language per source file; skip langdetect re-filtering
        'corpus_a_max_bytes': 20_000_000,
    },
    'expand': {
        'enabled': True,
        'add_vocab_size': 8000,
        'algorithm': 'bpe',
    },
    'cpt': {
        'enabled': True,
        'use_lora': True,
        'num_train_epochs': 1.0,
        'max_steps': -1,
        'per_device_train_batch_size': 8,   # A100 headroom -- raise further if you like
        'learning_rate': 2.0e-4,
    },
    'sft': {
        'enabled': True,
        'data_path': 'data/sft_data.jsonl',
        'use_lora': True,
        'num_train_epochs': 2.0,
        'per_device_train_batch_size': 8,
        'learning_rate': 1.0e-5,
    },
    'dpo': {
        'enabled': True,
        'data_path': 'data/dpo_data.jsonl',
        'use_lora': True,
        'num_train_epochs': 1.0,
        'per_device_train_batch_size': 4,
        'learning_rate': 5.0e-7,
    },
    'evaluate': {
        'enabled': True,
        'perplexity_text_file': 'data/eval.txt',
    },
    'export': {
        'enabled': True,
        'gguf': True,
        'gguf_outtype': 'f16',
        'gguf_pre_tokenizer_hint': 'qwen2',
        'onnx': True,
        'onnx_task': 'text-generation-with-past',
    },
}

Path('pipeline_config.yaml').write_text(yaml.dump(config, sort_keys=False), encoding='utf-8')
print(yaml.dump(config, sort_keys=False))

## 7. Run the full pipeline

This is the long cell. On an A100 with the defaults above, expect the CPT/SFT/DPO stages to be the bulk of the
wall-clock time; GGUF/ONNX export of a 1.5B model also takes a few minutes each.

In [ ]:
!python -m tok_adapt.cli pipeline --config pipeline_config.yaml

## 8. Inspect the results

In [ ]:
import json
summary = json.loads(Path('pipeline_out/summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))

## 9. Sanity-check generation quality

Not a benchmark — just eyeballing that the final DPO checkpoint produces coherent, in-language continuations
for a couple of the languages in the corpus.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

final_dir = summary['final_model_path']
gen_tokenizer = AutoTokenizer.from_pretrained(final_dir)
gen_model = AutoModelForCausalLM.from_pretrained(final_dir).to('cuda')

prompts = [
    'The history of artificial intelligence',
    lang_paragraphs['hi'][0].split(' ')[0] if lang_paragraphs.get('hi') else 'नमस्ते',
]
for prompt in prompts:
    inputs = gen_tokenizer(prompt, return_tensors='pt').to('cuda')
    out = gen_model.generate(**inputs, max_new_tokens=40, do_sample=False)
    print('PROMPT:', prompt)
    print('OUTPUT:', gen_tokenizer.decode(out[0], skip_special_tokens=True))
    print()